<a href="https://colab.research.google.com/github/aayamtiwari123/AI-concepts-/blob/main/Lab7_2_Encoder_Decoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RNN Encoder-Decoder for Statistical Machine Translation

## Setup

In [23]:
from __future__ import unicode_literals, print_function, division
from io import open
import unicodedata
import re
import random

import torch
import torch.nn as nn
from torch import optim
import torch.nn.functional as F

import numpy as np
from torch.utils.data import TensorDataset, DataLoader, RandomSampler

# Set the device to CUDA if available, otherwise use CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Set PyTorch's default device
torch.set_default_device(device)
print(f"Using device = {torch.get_default_device()}")

Using device = cpu


## Loading Data

Download from the following link:
- [https://www.manythings.org/anki/](https://download.pytorch.org/tutorial/data.zip)

Then, extract the data: `data/eng-fra.txt`. The file is a tab separated list of translation pairs.

In [24]:
import os
import zipfile
import requests

# URL of the dataset zip file
data_url = "https://download.pytorch.org/tutorial/data.zip"
zip_file_name = "data.zip"
data_dir = "data"

# Create data directory if it doesn't exist
os.makedirs(data_dir, exist_ok=True)

# Download the zip file
print(f"Downloading {data_url}...")
response = requests.get(data_url)
with open(zip_file_name, "wb") as f:
    f.write(response.content)
print(f"Downloaded {zip_file_name}.")

# Extract the zip file
print(f"Extracting {zip_file_name}...")
with zipfile.ZipFile(zip_file_name, 'r') as zip_ref:
    zip_ref.extractall(data_dir)
print(f"Extracted contents to {data_dir}.")

# Remove the zip file after extraction
os.remove(zip_file_name)
print(f"Removed {zip_file_name}.")

Downloaded data.zip.
Extracting data.zip...
Extracted contents to data.
Removed data.zip.


In [25]:
print('Dataset download and extraction complete. You can now re-run the data loading and training cells.')

Dataset download and extraction complete. You can now re-run the data loading and training cells.


In [26]:
SOS_token = 0 # Start of the Sentence token
EOS_token = 1 # End of the Sentence token

class Lang:
    """Helper class to manage vocabulary (word to index, index to word, and word counts)."""
    def __init__(self, name):
        self.name = name
        self.word2index = {} # Maps words to their integer indices
        self.word2count = {} # Counts the occurrences of each word
        self.index2word = {0: "SOS", 1: "EOS"} # Maps integer indices back to words, initialized with SOS and EOS
        self.n_words = 2  # Count of unique words, starting with SOS and EOS

    def addSentence(self, sentence):
        """Adds all words in a sentence to the vocabulary."""
        for word in sentence.split(' '):
            self.addWord(word)

    def addWord(self, word):
        """Adds a single word to the vocabulary or increments its count."""
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.word2count[word] = 1
            self.index2word[self.n_words] = word
            self.n_words += 1
        else:
            self.word2count[word] += 1

In [27]:
# Turn a Unicode string to plain ASCII, thanks to
# https://stackoverflow.com/a/518232/2809427
def unicodeToAscii(s):
    """Converts unicode string to plain ASCII."""
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
    )

# Lowercase, trim, and remove non-letter characters
def normalizeString(s):
    """Normalizes a string by lowercasing, trimming, and cleaning non-alphanumeric characters."""
    s = unicodeToAscii(s.lower().strip())
    s = re.sub(r"([.!?])", r" \1", s) # Add space before punctuation
    s = re.sub(r"[^a-zA-Z!?]+", r" ", s) # Replace non-alphanumeric with spaces
    return s.strip()

In [28]:
def readLangs(path:str):
    """Reads a language pair dataset from a file, normalizes it, and creates Lang objects."""
    lang1 = 'eng'; lang2 = 'fra'
    print("Reading lines...")

    # Read the file and split into lines
    lines = open(path, encoding='utf-8').\
        read().strip().split('\n')

    # Split every line into pairs and normalize (english to french)
    pairs = [[normalizeString(s) for s in l.split('\t')] for l in lines]

    # Reverse pairs: English-French -> French-English
    # The input language for our model will be French, and output will be English.
    pairs = [list(reversed(p)) for p in pairs]

    # Create Lang objects for input (French) and output (English)
    input_lang = Lang(lang2)
    output_lang = Lang(lang1)

    return input_lang, output_lang, pairs

## Data Preprocessing Utilities

In [29]:
MAX_LENGTH = 5 # Maximum sentence length for filtering

# English prefixes to filter target sentences
eng_prefixes = (
    "i am ", "i m ",
    "he is", "he s ",
    "she is", "she s ",
    "you are", "you re ",
    "we are", "we re ",
    "they are", "they re "
)

def filterPair(p):
    """Filters a single sentence pair based on length and English prefix."""
    return len(p[0].split(' ')) < MAX_LENGTH and \
        len(p[1].split(' ')) < MAX_LENGTH and \
        p[1].startswith(eng_prefixes)


def filterPairs(pairs):
    """Filters a list of sentence pairs using the filterPair function."""
    return [pair for pair in pairs if filterPair(pair)]

In [30]:
def prepareData(path):
    """Reads, filters, and prepares data for training by creating vocabulary and pairs."""
    input_lang, output_lang, pairs = readLangs(path)
    print("Read %s sentence pairs" % len(pairs))
    pairs = filterPairs(pairs)
    print("Trimmed to %s sentence pairs" % len(pairs))
    print("Counting words...")
    for pair in pairs:
        input_lang.addSentence(pair[0])
        output_lang.addSentence(pair[1])
    print("Counted words:")
    print(input_lang.name, input_lang.n_words)
    print(output_lang.name, output_lang.n_words)
    return input_lang, output_lang, pairs

In [31]:
# Define the path to the dataset file
PATH = r'data/data/eng-fra.txt'

# Prepare the data: read, filter, and build vocabulary
input_lang, output_lang, pairs = prepareData(PATH)

# Print a random sentence pair from the prepared dataset
print(random.choice(pairs))

# Example: Look up the index of the word 'am' in the output language vocabulary
output_lang.word2index['am']

Reading lines...
Read 135842 sentence pairs
Trimmed to 3272 sentence pairs
Counting words...
Counted words:
fra 1757
eng 967
['il est tres malade', 'he s very ill']


15

## Encoder-Decoder Network

### Encoder

In [32]:
class EncoderRNN(nn.Module):
    """The Encoder RNN processes the input sequence and produces a context vector."""
    def __init__(self, input_size, hidden_size, dropout_p=0.1):
        super(EncoderRNN, self).__init__()
        self.hidden_size = hidden_size

        # Embedding layer to convert word indices to dense vectors
        self.embedding = nn.Embedding(input_size, hidden_size)
        # RNN layer (e.g., GRU or LSTM can also be used) which processes the sequence
        self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)
        # Dropout layer for regularization to prevent overfitting
        self.dropout = nn.Dropout(dropout_p)

    def forward(self, input):
        """Forward pass for the encoder.
        Args:
            input (torch.Tensor): Input tensor of word indices.
        Returns:
            tuple: Output and hidden state of the RNN.
        """
        # Apply dropout to the embedded input
        embedded = self.dropout(self.embedding(input))
        # Pass the embedded input through the RNN
        output, hidden = self.rnn(embedded)
        return output, hidden

### Decoder

In [33]:
class DecoderRNN(nn.Module):
    """The Decoder RNN takes the encoder's output and generates an output sequence."""
    def __init__(self, hidden_size, output_size):
        super(DecoderRNN, self).__init__()
        # Embedding layer for output tokens
        self.embedding = nn.Embedding(output_size, hidden_size)
        # RNN layer for sequence generation
        self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)
        # Linear layer to project RNN output to vocabulary size
        self.out = nn.Linear(hidden_size, output_size)

    def forward(self, encoder_outputs, encoder_hidden, target_tensor=None):
        """Forward pass for the decoder.
        Args:
            encoder_outputs (torch.Tensor): Outputs from the encoder.
            encoder_hidden (torch.Tensor): Final hidden state from the encoder (context vector).
            target_tensor (torch.Tensor, optional): Ground truth target sequence for teacher forcing.
        Returns:
            tuple: Decoder outputs, final hidden state, and attention (None in this basic model).
        """
        batch_size = encoder_outputs.size(0)
        # Initialize decoder input with SOS token for each item in the batch
        decoder_input = torch.empty(batch_size, 1, dtype=torch.long, device=device).fill_(SOS_token)
        # Initialize decoder hidden state with the encoder's final hidden state
        decoder_hidden = encoder_hidden
        decoder_outputs = []

        # Loop through MAX_LENGTH to generate the output sequence
        for i in range(MAX_LENGTH):
            decoder_output, decoder_hidden  = self.forward_step(decoder_input, decoder_hidden)
            decoder_outputs.append(decoder_output)

            if target_tensor is not None:
                # Teacher forcing: Feed the target token as the next input
                decoder_input = target_tensor[:, i].unsqueeze(1) # Teacher forcing
            else:
                # Without teacher forcing: use the decoder's own prediction as the next input
                _, topi = decoder_output.topk(1) # Get the index of the highest-scoring word
                decoder_input = topi.squeeze(-1).detach()  # Detach to prevent gradients flowing back through generated input

        # Concatenate all decoder outputs and apply log_softmax for probabilities
        decoder_outputs = torch.cat(decoder_outputs, dim=1)
        decoder_outputs = F.log_softmax(decoder_outputs, dim=-1)
        # Return outputs, final hidden state, and None for attention (as not implemented here)
        return decoder_outputs, decoder_hidden, None

    def forward_step(self, input, hidden):
        """Performs a single decoding step.
        Args:
            input (torch.Tensor): Current input token (index).
            hidden (torch.Tensor): Current hidden state.
        Returns:
            tuple: Output of the step and new hidden state.
        """
        # Embed the input token
        output = self.embedding(input)
        # Apply ReLU activation
        output = F.relu(output)
        # Pass through the RNN layer
        output, hidden = self.rnn(output, hidden)
        # Project RNN output to vocabulary size
        output = self.out(output)
        return output, hidden

## Training

## Prepare Training Data

In [34]:
def indexesFromSentence(lang, sentence):
    """Converts a sentence string to a list of word indices using the given language's vocabulary."""
    return [lang.word2index[word] for word in sentence.split(' ')]

def tensorFromSentence(lang, sentence):
    """Converts a sentence string to a PyTorch tensor of word indices, appending EOS token."""
    indexes = indexesFromSentence(lang, sentence)
    indexes.append(EOS_token)
    # Create a tensor and reshape for model input
    return torch.tensor(indexes, dtype=torch.long, device=device).view(1, -1)

def tensorsFromPair(pair):
    """Converts a language pair (list of two sentences) into input and target tensors."""
    input_tensor = tensorFromSentence(input_lang, pair[0])
    target_tensor = tensorFromSentence(output_lang, pair[1])
    return (input_tensor, target_tensor)

def get_dataloader(batch_size):
    """Prepares the DataLoader for training, including loading data, filtering, and tensor conversion."""
    input_lang, output_lang, pairs = prepareData(path=PATH)

    n = len(pairs)
    # Initialize numpy arrays to hold padded input and target IDs
    input_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)
    target_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)

    for idx, (inp, tgt) in enumerate(pairs):
        # Convert sentences to lists of word indices
        inp_ids = indexesFromSentence(input_lang, inp)
        tgt_ids = indexesFromSentence(output_lang, tgt)
        inp_ids.append(EOS_token) # Append EOS token
        tgt_ids.append(EOS_token) # Append EOS token
        # Pad sequences and store in numpy arrays
        input_ids[idx, :len(inp_ids)] = inp_ids
        target_ids[idx, :len(tgt_ids)] = tgt_ids

    # Create a TensorDataset from the prepared tensors
    train_data = TensorDataset(torch.LongTensor(input_ids).to(device),
                               torch.LongTensor(target_ids).to(device))

    # Create a RandomSampler for shuffling data
    train_sampler = RandomSampler(train_data)
    # Create a DataLoader for batching data during training
    train_dataloader = DataLoader(train_data, sampler=train_sampler, batch_size=batch_size)
    return input_lang, output_lang, train_dataloader

### Training Loop

In [35]:
def train_epoch(dataloader, encoder, decoder, encoder_optimizer,
          decoder_optimizer, criterion):
    """Performs one full training epoch over the provided dataloader."""
    total_loss = 0
    for data in dataloader:
        input_tensor, target_tensor = data

        # Zero gradients for both optimizers
        encoder_optimizer.zero_grad()
        decoder_optimizer.zero_grad()

        # Forward pass through the encoder
        encoder_outputs, encoder_hidden = encoder(input_tensor)
        # Forward pass through the decoder, using teacher forcing
        decoder_outputs, _, _ = decoder(encoder_outputs, encoder_hidden, target_tensor)

        # Calculate loss
        loss = criterion(
            decoder_outputs.view(-1, decoder_outputs.size(-1)), # Reshape decoder outputs for NLLLoss
            target_tensor.view(-1) # Reshape target tensor
        )
        # Backpropagate the loss
        loss.backward()

        # Perform a single optimization step
        encoder_optimizer.step()
        decoder_optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader) # Return average loss for the epoch

In [36]:
import time
import math

def asMinutes(s):
    """Converts seconds into a 'minutes:seconds' string format."""
    m = math.floor(s / 60)
    s -= m * 60
    return '%dm %ds' % (m, s)

def timeSince(since, percent):
    """Calculates time elapsed and estimated remaining time during training."""
    now = time.time()
    s = now - since # Seconds elapsed
    es = s / (percent) # Estimated total time
    rs = es - s # Estimated remaining time
    return '%s (- %s)' % (asMinutes(s), asMinutes(rs))

In [37]:
import matplotlib.pyplot as plt
# Set the backend for matplotlib. 'agg' is non-interactive, suitable for servers or scripts.
plt.switch_backend('agg')
import matplotlib.ticker as ticker
import numpy as np

def showPlot(points):
    """Generates and displays a plot of training loss over time."""
    plt.figure()
    fig, ax = plt.subplots()
    # This locator puts ticks at regular intervals on the y-axis
    loc = ticker.MultipleLocator(base=0.2)
    ax.yaxis.set_major_locator(loc)
    plt.plot(points)
    plt.title('Training Loss Over Epochs') # Add a title to the plot
    plt.xlabel('Epoch (scaled by plot_every)') # Add x-axis label
    plt.ylabel('Average Loss') # Add y-axis label
    # The plot won't be explicitly displayed if backend is 'agg', but can be saved or embedded.

In [38]:
def train(train_dataloader, encoder, decoder, n_epochs, learning_rate=0.001,
               print_every=100, plot_every=100):
    """Main training function to run multiple epochs and track progress."""
    start = time.time()
    plot_losses = [] # List to store losses for plotting
    print_loss_total = 0  # Accumulator for losses to print periodically
    plot_loss_total = 0  # Accumulator for losses to plot periodically

    # Initialize optimizers for encoder and decoder with a specified learning rate
    encoder_optimizer = optim.Adam(encoder.parameters(), lr=learning_rate)
    decoder_optimizer = optim.Adam(decoder.parameters(), lr=learning_rate)
    # Define the loss function (Negative Log Likelihood Loss)
    criterion = nn.NLLLoss()

    # Iterate through the specified number of epochs
    for epoch in range(1, n_epochs + 1):
        # Train for one epoch and get the average loss
        loss = train_epoch(train_dataloader, encoder, decoder, encoder_optimizer, decoder_optimizer, criterion)
        print_loss_total += loss
        plot_loss_total += loss

        # Print training progress every 'print_every' epochs
        if epoch % print_every == 0:
            print_loss_avg = print_loss_total / print_every
            print_loss_total = 0
            print('%s (%d %d%%) %.4f' % (timeSince(start, epoch / n_epochs),
                                        epoch, epoch / n_epochs * 100, print_loss_avg))

        # Store average loss for plotting every 'plot_every' epochs
        if epoch % plot_every == 0:
            plot_loss_avg = plot_loss_total / plot_every
            plot_losses.append(plot_loss_avg)
            plot_loss_total = 0

    # Display the plot of accumulated losses after training is complete
    showPlot(plot_losses)

## Evaluation Code

In [39]:
def evaluate(encoder, decoder, sentence, input_lang, output_lang):
    """Evaluates the model on a single input sentence."""
    # Disable gradient calculations during evaluation
    with torch.no_grad():
        # Convert the input sentence to a tensor of indices
        input_tensor = tensorFromSentence(input_lang, sentence)

        # Pass the input through the encoder
        encoder_outputs, encoder_hidden = encoder(input_tensor)
        # Pass encoder outputs and hidden state to the decoder
        decoder_outputs, decoder_hidden, decoder_attn = decoder(encoder_outputs, encoder_hidden)

        # Get the top predicted word index for each step
        _, topi = decoder_outputs.topk(1)
        decoded_ids = topi.squeeze()

        decoded_words = []
        for idx in decoded_ids:
            if idx.item() == EOS_token:
                decoded_words.append('<EOS>')
                break
            # Convert index back to word and append
            decoded_words.append(output_lang.index2word[idx.item()])
    return decoded_words, decoder_attn

In [40]:
def evaluateRandomly(encoder, decoder, n=10):
    """Evaluates the model on 'n' random sentence pairs from the dataset."""
    for i in range(n):
        pair = random.choice(pairs) # Select a random pair
        print('>', pair[0]) # Print input sentence
        print('=', pair[1]) # Print target sentence
        # Evaluate the model and get the predicted output words
        output_words, _ = evaluate(encoder, decoder, pair[0], input_lang, output_lang)
        output_sentence = ' '.join(output_words)
        print('<', output_sentence) # Print predicted output sentence
        print('')

### Training and Evaluating

In [41]:
# Define model parameters
hidden_size = 128 # Size of the hidden layers in RNNs
batch_size = 32   # Number of samples per batch
EPOCHS = 200      # Number of training epochs

# Get the DataLoader and language objects
input_lang, output_lang, train_dataloader = get_dataloader(batch_size)

# Initialize Encoder and Decoder models and move them to the specified device
encoder = EncoderRNN(input_lang.n_words, hidden_size).to(device)
decoder = DecoderRNN(hidden_size, output_lang.n_words).to(device)

# Start the training process
train(train_dataloader, encoder, decoder, EPOCHS, print_every=5, plot_every=5)

Reading lines...
Read 135842 sentence pairs
Trimmed to 3272 sentence pairs
Counting words...
Counted words:
fra 1757
eng 967
0m 7s (- 4m 36s) (5 2%) 1.9498
0m 13s (- 4m 9s) (10 5%) 1.2871
0m 20s (- 4m 7s) (15 7%) 1.0933
0m 26s (- 3m 54s) (20 10%) 0.9573
0m 32s (- 3m 45s) (25 12%) 0.8468
0m 38s (- 3m 36s) (30 15%) 0.7444
0m 44s (- 3m 31s) (35 17%) 0.6497
0m 50s (- 3m 23s) (40 20%) 0.5634
0m 57s (- 3m 16s) (45 22%) 0.4898
1m 3s (- 3m 11s) (50 25%) 0.4225
1m 11s (- 3m 8s) (55 27%) 0.3684
1m 19s (- 3m 5s) (60 30%) 0.3216
1m 25s (- 2m 58s) (65 32%) 0.2835
1m 32s (- 2m 51s) (70 35%) 0.2474
1m 39s (- 2m 45s) (75 37%) 0.2189
1m 45s (- 2m 37s) (80 40%) 0.1937
1m 51s (- 2m 31s) (85 42%) 0.1719
1m 58s (- 2m 24s) (90 45%) 0.1565
2m 4s (- 2m 17s) (95 47%) 0.1374
2m 10s (- 2m 10s) (100 50%) 0.1252
2m 17s (- 2m 4s) (105 52%) 0.1137
2m 24s (- 1m 57s) (110 55%) 0.1063
2m 30s (- 1m 50s) (115 57%) 0.0982
2m 36s (- 1m 44s) (120 60%) 0.0957
2m 43s (- 1m 38s) (125 62%) 0.0854
2m 50s (- 1m 31s) (130 65%) 0.0

In [42]:
encoder.eval()
decoder.eval()
evaluateRandomly(encoder, decoder)

> elles sont miennes
= they re mine
< you are morons <EOS>

> tu es tres marrant
= you re very funny
< you re very funny <EOS>

> tu es jalouse
= you re jealous
< we re really disappointed <EOS>

> vous etes tres vives
= you re very sharp
< you re very sharp <EOS>

> je suis vraiment heureuse
= i m really happy
< i m really happy <EOS>

> tu es tres indelicat
= you are very insensitive
< you are very insensitive <EOS>

> tu es vraiment egoiste
= you re really selfish
< you re really selfish <EOS>

> nous sommes tous aneantis
= we re all devastated
< we re all devastated <EOS>

> nous fermons tot
= we re closing early
< i m extremely happy <EOS>

> je suis assez heureux
= i m quite happy
< i m happy enough <EOS>

